# Third Molar Conditional GAN (CGAN)

This notebook implements a Conditional Generative Adversarial Network (CGAN) for generating synthetic third molar dental images based on different classification labels.

## Dataset
- **Third Molar Images**: 5,416 PNG images
- **Classes**: 7 different third molar classifications (0-6)
- **Format**: RGB images
- **Source**: `./thirdmolar/` directory

## Implementation
- Conditional Generator and Discriminator
- Label embedding for class conditioning
- RGB image support (3 channels)
- Batch normalization and dropout for stability

---

## Import Libraries

In [ ]:
%matplotlib inline
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torch import autograd
from torch.autograd import Variable
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('torch version:',torch.__version__)
print('device:', device)

## - Parameters

In [ ]:
# Data
train_data_path = './thirdmolar/'  # Path to third molar images
print('Train data path:', train_data_path)

# Binary classification option
binary_classification = False  # True: Class 0 vs Rest, False: All 7 classes
print(f'Binary classification (0 vs rest): {binary_classification}')

# Image parameters - need to determine from actual images
img_size = 128  # Updated to 128x128 for higher resolution
batch_size = 64  # Batch size

# Model parameters
z_size = 100
generator_layer_size = [256, 512, 1024]
discriminator_layer_size = [1024, 512, 256]

# Training parameters
epochs = 1000  # More epochs for medical data
learning_rate = 2e-4  # Slightly higher learning rate

## Dataset: Third Molar Images

Loading and preprocessing the third molar dental images with their corresponding classification labels.

In [ ]:
# Configure classes based on binary_classification setting
if binary_classification:
    # Binary classification: Class 0 vs Rest
    class_list = ['Class 0', 'Other Classes (1-6)']
    class_num = 2
    print("=== BINARY CLASSIFICATION MODE ===")
    print("Class 0: Original Class 0")
    print("Class 1: All other classes (1, 2, 3, 4, 5, 6)")
else:
    # Multi-class: All 7 original classes
    class_list = ['Class 0', 'Class 1', 'Class 2', 'Class 3', 'Class 4', 'Class 5', 'Class 6']
    class_num = 7
    print("=== MULTI-CLASS MODE ===")
    print("Using all 7 original classes")

print(f"Number of classes: {class_num}")
print("Classes:", class_list)

In [ ]:
import os
from collections import Counter

class ThirdMolarDataset(Dataset):
    def __init__(self, data_path, transform=None, binary_classification=False):
        self.data_path = data_path
        self.transform = transform
        self.binary_classification = binary_classification
        self.image_paths = []
        self.labels = []
        self.original_labels = []  # Keep track of original labels
        
        # Get all PNG files
        for filename in os.listdir(data_path):
            if filename.endswith('.png'):
                # Extract label from filename: {number}_{classification}_{label}.png
                try:
                    original_label = int(filename.split('_')[-1].split('.')[0])
                    self.image_paths.append(os.path.join(data_path, filename))
                    self.original_labels.append(original_label)
                    
                    # Convert to binary if needed
                    if self.binary_classification:
                        # Class 0 -> 0, Classes 1-6 -> 1
                        binary_label = 0 if original_label == 0 else 1
                        self.labels.append(binary_label)
                    else:
                        self.labels.append(original_label)
                        
                except (ValueError, IndexError):
                    print(f"Skipping file with invalid name format: {filename}")
        
        # Print dataset statistics
        print(f'Total images loaded: {len(self.image_paths)}')
        
        if self.binary_classification:
            print('Binary classification - Label distribution:')
            label_counts = Counter(self.labels)
            original_counts = Counter(self.original_labels)
            print(f'  Class 0 (Original Class 0): {label_counts[0]} images')
            print(f'  Class 1 (Original Classes 1-6): {label_counts[1]} images')
            print('  Original distribution within "Other Classes":')
            for orig_label in sorted([l for l in original_counts.keys() if l != 0]):
                print(f'    Original Class {orig_label}: {original_counts[orig_label]} images')
        else:
            print('Multi-class - Label distribution:')
            label_counts = Counter(self.labels)
            for label in sorted(label_counts.keys()):
                print(f'  Class {label}: {label_counts[label]} images')
            
        # Check image size from first image
        if self.image_paths:
            first_img = Image.open(self.image_paths[0])
            self.img_width, self.img_height = first_img.size
            print(f'Image dimensions: {self.img_width} x {self.img_height}')
            first_img.close()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        # Load image
        img = Image.open(img_path).convert('RGB')  # Ensure RGB format
        
        if self.transform:
            img = self.transform(img)
        
        return img, label

In [ ]:
# Load dataset to check image dimensions
temp_dataset = ThirdMolarDataset(train_data_path, binary_classification=binary_classification)

# Update img_size based on actual images
if hasattr(temp_dataset, 'img_width') and hasattr(temp_dataset, 'img_height'):
    # Use the smaller dimension or resize to square
    img_size = min(temp_dataset.img_width, temp_dataset.img_height)
    print(f"Setting image size to: {img_size}")
else:
    img_size = 64  # fallback
    print(f"Using fallback image size: {img_size}")

In [ ]:
# Test loading first image
img, label = temp_dataset[0]
print(f"First image shape: {img.size}")
print(f"First image label: {label} ({class_list[label]})")
img

In [ ]:
# Show another image
img_10, label_10 = temp_dataset[10]
print(f"Image 10 shape: {img_10.size}")
print(f"Image 10 label: {label_10} ({class_list[label_10]})")
img_10

In [ ]:
# Define transforms for the dataset
transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),  # Resize to square
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))  # Normalize for RGB
])

print(f"Transform setup for image size: {img_size}x{img_size}")
print("Normalization: RGB channels normalized to [-1, 1]")

In [ ]:
# Create final dataset with transforms
dataset = ThirdMolarDataset(train_data_path, transform=transform, binary_classification=binary_classification)
data_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"Dataset size: {len(dataset)}")
print(f"Number of batches: {len(data_loader)}")
print(f"Batch size: {batch_size}")

if binary_classification:
    print("\n=== BINARY CLASSIFICATION SUMMARY ===")
    print("Class 0: Third molars from original Class 0")
    print("Class 1: Third molars from original Classes 1-6 combined")
else:
    print("\n=== MULTI-CLASS SUMMARY ===")
    print("Using all 7 original classes")

In [ ]:
# Visualize a batch of images
for images, labels in data_loader:
    fig, ax = plt.subplots(figsize=(18,10))
    ax.set_xticks([])
    ax.set_yticks([])
    # Create grid and normalize for display
    grid = make_grid(images[:16], nrow=4, normalize=True)
    ax.imshow(grid.permute(1,2,0))
    
    # Print labels for this batch
    print("Labels in this batch:", labels[:16].tolist())
    print("Classes:", [class_list[label.item()] for label in labels[:16]])
    break

## - Generator

In [ ]:
class Generator(nn.Module):
    def __init__(self, generator_layer_size, z_size, img_size, class_num):
        super().__init__()
        
        self.z_size = z_size
        self.img_size = img_size
        self.channels = 3  # RGB images
        
        self.label_emb = nn.Embedding(class_num, class_num)
        
        self.model = nn.Sequential(
            nn.Linear(self.z_size + class_num, generator_layer_size[0]),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(generator_layer_size[0]),
            nn.Linear(generator_layer_size[0], generator_layer_size[1]),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(generator_layer_size[1]),
            nn.Linear(generator_layer_size[1], generator_layer_size[2]),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(generator_layer_size[2]),
            nn.Linear(generator_layer_size[2], self.img_size * self.img_size * self.channels),
            nn.Tanh()
        )
    
    def forward(self, z, labels):
        # Reshape z
        z = z.view(-1, self.z_size)
        
        # One-hot vector to embedding vector
        c = self.label_emb(labels)
        
        # Concat noise & label
        x = torch.cat([z, c], 1)
        
        # Generator output
        out = self.model(x)
        
        # Reshape to image format: (batch_size, channels, height, width)
        return out.view(-1, self.channels, self.img_size, self.img_size)

## - Discriminator

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, discriminator_layer_size, img_size, class_num):
        super().__init__()
        
        self.label_emb = nn.Embedding(class_num, class_num)
        self.img_size = img_size
        self.channels = 3  # RGB images
        
        self.model = nn.Sequential(
            nn.Linear(self.img_size * self.img_size * self.channels + class_num, discriminator_layer_size[0]),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(discriminator_layer_size[0], discriminator_layer_size[1]),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(discriminator_layer_size[1], discriminator_layer_size[2]),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(discriminator_layer_size[2], 1),
            nn.Sigmoid()
        )
    
    def forward(self, x, labels):
        # Reshape image to flat vector
        x = x.view(-1, self.img_size * self.img_size * self.channels)
        
        # One-hot vector to embedding vector
        c = self.label_emb(labels)
        
        # Concat image & label
        x = torch.cat([x, c], 1)
        
        # Discriminator output
        out = self.model(x)
        
        return out.squeeze()

In [ ]:
# Define generator
generator = Generator(generator_layer_size, z_size, img_size, class_num).to(device)
# Define discriminator
discriminator = Discriminator(discriminator_layer_size, img_size, class_num).to(device)

In [ ]:
# Model information and parameter counts
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=== Model Architecture Summary ===")
print(f"Generator parameters: {count_parameters(generator):,}")
print(f"Discriminator parameters: {count_parameters(discriminator):,}")
print(f"Total parameters: {count_parameters(generator) + count_parameters(discriminator):,}")

print(f"\nImage size: {img_size}x{img_size} RGB")
print(f"Total pixels per image: {img_size * img_size * 3:,}")

# Test forward pass
with torch.no_grad():
    # Test generator
    test_z = torch.randn(2, z_size).to(device)
    test_labels = torch.LongTensor([0, 1]).to(device)
    fake_images = generator(test_z, test_labels)
    print(f"\nGenerator output shape: {fake_images.shape}")
    
    # Test discriminator
    pred = discriminator(fake_images, test_labels)
    print(f"Discriminator output shape: {pred.shape}")

print("\nModels successfully created and tested!")

In [ ]:
# Using 128x128 images for higher quality generation
print(f"Current model size: {count_parameters(generator) + count_parameters(discriminator):,} parameters")
print("Training with 128x128 images for better quality synthetic third molars.")

# Use 128x128 image size for training
img_size_high = 128

# Recreate models with 128x128 image size
generator_high = Generator(generator_layer_size, z_size, img_size_high, class_num).to(device)
discriminator_high = Discriminator(discriminator_layer_size, img_size_high, class_num).to(device)

high_params = count_parameters(generator_high) + count_parameters(discriminator_high)
print(f"128x128 model size: {high_params:,} parameters")

# Use the higher resolution version for training
img_size = img_size_high
generator = generator_high
discriminator = discriminator_high

print(f"\nUsing {img_size}x{img_size} image size for high-quality training")

In [ ]:
# Recreate dataset and dataloader with 128x128 image size
transform_high = transforms.Compose([
    transforms.Resize((img_size, img_size)),  # Resize to 128x128
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))  # Normalize for RGB
])

dataset = ThirdMolarDataset(train_data_path, transform=transform_high, binary_classification=binary_classification)
data_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"Updated dataset for {img_size}x{img_size} images")
print(f"Dataset size: {len(dataset)}")
print(f"Number of batches: {len(data_loader)}")

if binary_classification:
    print("\n=== BINARY MODE ACTIVE ===")
    print("Training CGAN for: Class 0 vs Other Classes")
else:
    print("\n=== MULTI-CLASS MODE ACTIVE ===")
    print("Training CGAN for: All 7 classes")

print(f"\nHigh-resolution 128x128 training enabled for better quality synthetic images!")

## - Adversarial Learning of Generator & Discriminator

In [ ]:
# Loss function
criterion = nn.BCELoss()

In [ ]:
# Optimizer
g_optimizer = torch.optim.Adam(generator.parameters(), lr=learning_rate)
d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=learning_rate)

In [ ]:
def generator_train_step(batch_size, discriminator, generator, g_optimizer, criterion):
    
    # Init gradient
    g_optimizer.zero_grad()
    
    # Building z
    z = Variable(torch.randn(batch_size, z_size)).to(device)
    
    # Building fake labels
    fake_labels = Variable(torch.LongTensor(np.random.randint(0, class_num, batch_size))).to(device)
    
    # Generating fake images
    fake_images = generator(z, fake_labels)
    
    # Disciminating fake images
    validity = discriminator(fake_images, fake_labels)
    
    # Calculating discrimination loss (fake images)
    g_loss = criterion(validity, Variable(torch.ones(batch_size)).to(device))
    
    # Backword propagation
    g_loss.backward()
    
    #  Optimizing generator
    g_optimizer.step()
    
    return g_loss.data

In [ ]:
def discriminator_train_step(batch_size, discriminator, generator, d_optimizer, criterion, real_images, labels):
    
    # Init gradient 
    d_optimizer.zero_grad()

    # Disciminating real images
    real_validity = discriminator(real_images, labels)
    
    # Calculating discrimination loss (real images)
    real_loss = criterion(real_validity, Variable(torch.ones(batch_size)).to(device))
    
    # Building z
    z = Variable(torch.randn(batch_size, z_size)).to(device)
    
    # Building fake labels
    fake_labels = Variable(torch.LongTensor(np.random.randint(0, class_num, batch_size))).to(device)
    
    # Generating fake images
    fake_images = generator(z, fake_labels)
    
    # Disciminating fake images
    fake_validity = discriminator(fake_images, fake_labels)
    
    # Calculating discrimination loss (fake images)
    fake_loss = criterion(fake_validity, Variable(torch.zeros(batch_size)).to(device))
    
    # Sum two losses
    d_loss = real_loss + fake_loss
    
    # Backword propagation
    d_loss.backward()
    
    # Optimizing discriminator
    d_optimizer.step()
    
    return d_loss.data

In [ ]:
# Quick test of training functions
print("Testing training functions...")

# Get a small batch for testing
test_images, test_labels = next(iter(data_loader))
test_images = Variable(test_images[:8]).to(device)  # Use only 8 images for test
test_labels = Variable(test_labels[:8]).to(device)

# Test discriminator training step
d_loss_test = discriminator_train_step(len(test_images), discriminator, 
                                     generator, d_optimizer, criterion,
                                     test_images, test_labels)

# Test generator training step  
g_loss_test = generator_train_step(len(test_images), discriminator, 
                                 generator, g_optimizer, criterion)

print(f"Test D_loss: {d_loss_test:.4f}")
print(f"Test G_loss: {g_loss_test:.4f}")
print("Training functions working correctly!")

# Test image generation
generator.eval()
with torch.no_grad():
    test_z = Variable(torch.randn(class_num, z_size)).to(device)
    test_gen_labels = Variable(torch.LongTensor(list(range(class_num)))).to(device)
    test_generated = generator(test_z, test_gen_labels)
    print(f"Generated images shape: {test_generated.shape}")
    
print("Ready for full training!")

In [ ]:
# Training loop
generator_losses = []
discriminator_losses = []

for epoch in range(epochs):
    print(f'Starting epoch {epoch+1}/{epochs}...')
    
    epoch_g_loss = 0
    epoch_d_loss = 0
    num_batches = 0
    
    for i, (images, labels) in enumerate(data_loader):
        # Train data
        real_images = Variable(images).to(device)
        labels = Variable(labels).to(device)
        
        # Set generator train
        generator.train()
        
        # Train discriminator
        d_loss = discriminator_train_step(len(real_images), discriminator,
                                          generator, d_optimizer, criterion,
                                          real_images, labels)
        
        # Train generator
        g_loss = generator_train_step(len(real_images), discriminator, generator, g_optimizer, criterion)
        
        epoch_g_loss += g_loss.item()
        epoch_d_loss += d_loss.item()
        num_batches += 1
        
        # Print progress every 50 batches
        if (i + 1) % 50 == 0:
            print(f'  Batch {i+1}/{len(data_loader)}: G_loss: {g_loss:.4f}, D_loss: {d_loss:.4f}')
    
    # Calculate average losses
    avg_g_loss = epoch_g_loss / num_batches
    avg_d_loss = epoch_d_loss / num_batches
    
    generator_losses.append(avg_g_loss)
    discriminator_losses.append(avg_d_loss)
    
    # Set generator eval
    generator.eval()
    
    print(f'Epoch {epoch+1} completed - G_loss: {avg_g_loss:.4f}, D_loss: {avg_d_loss:.4f}')
    
    # Generate sample images every 5 epochs
    if (epoch + 1) % 5 == 0:
        with torch.no_grad():
            # Generate one image per class
            z = Variable(torch.randn(class_num, z_size)).to(device)
            sample_labels = Variable(torch.LongTensor(list(range(class_num)))).to(device)
            
            # Generate images
            sample_images = generator(z, sample_labels).data.cpu()
            
            # Show images
            fig, axes = plt.subplots(1, class_num, figsize=(2*class_num, 2))
            for j in range(class_num):
                if class_num == 1:
                    ax = axes
                else:
                    ax = axes[j]
                img = sample_images[j].permute(1, 2, 0)
                img = (img + 1) / 2  # Denormalize from [-1,1] to [0,1]
                ax.imshow(img)
                ax.set_title(f'{class_list[j]}')
                ax.axis('off')
            plt.tight_layout()
            plt.show()

print("Training completed!")

## - Show Generating Images

In [ ]:
# Generate final showcase of images
generator.eval()

with torch.no_grad():
    # Generate multiple samples per class
    samples_per_class = 5
    total_samples = class_num * samples_per_class
    
    # Create noise and labels
    z = Variable(torch.randn(total_samples, z_size)).to(device)
    labels = Variable(torch.LongTensor([i for i in range(class_num) for _ in range(samples_per_class)])).to(device)
    
    # Generate images
    sample_images = generator(z, labels).data.cpu()
    
    # Denormalize images
    sample_images = (sample_images + 1) / 2  # From [-1,1] to [0,1]
    
    # Create visualization
    fig, axes = plt.subplots(class_num, samples_per_class, figsize=(samples_per_class*2, class_num*2))
    
    for i in range(class_num):
        for j in range(samples_per_class):
            idx = i * samples_per_class + j
            if class_num == 1:
                ax = axes[j] if samples_per_class > 1 else axes
            else:
                ax = axes[i, j] if samples_per_class > 1 else axes[i]
            
            img = sample_images[idx].permute(1, 2, 0)
            ax.imshow(img)
            if j == 0:  # Only show class name on first column
                ax.set_ylabel(f'{class_list[i]}', rotation=0, ha='right')
            ax.set_xticks([])
            ax.set_yticks([])
    
    plt.suptitle('Generated Third Molar Images by Class', fontsize=16)
    plt.tight_layout()
    plt.show()

# Plot training losses
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(generator_losses, label='Generator Loss')
plt.plot(discriminator_losses, label='Discriminator Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Losses')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(generator_losses, label='Generator Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Generator Loss Over Time')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Summary and next steps
print("=== Third Molar CGAN Summary ===")
print(f"Dataset: {len(dataset)} third molar images")
print(f"Classes: {class_num} different classifications")
print(f"Image size: {img_size}x{img_size} RGB")
print(f"Training epochs: {epochs}")
print(f"Batch size: {batch_size}")
print("\nClass distribution in dataset:")
from collections import Counter
label_counts = Counter(dataset.labels)
for label in sorted(label_counts.keys()):
    print(f"  {class_list[label]}: {label_counts[label]} images ({label_counts[label]/len(dataset)*100:.1f}%)")

print("\nModel Architecture:")
print("Generator: Fully connected layers with batch normalization")
print("Discriminator: Fully connected layers with dropout")
print("Conditioning: Label embedding concatenated with noise/image")

print("\nNext steps:")
print("1. Experiment with different architectures (CNN-based)")
print("2. Adjust hyperparameters for better convergence")
print("3. Implement evaluation metrics (FID, IS)")
print("4. Data augmentation for minority classes")
print("5. Progressive growing for higher resolution")

In [ ]:
# Synthesize 550 high-resolution 128x128 images for each class
import os
import random
from torchvision.utils import save_image

# Create synthesized directory if it doesn't exist
synthesized_dir = './synthesized/'
os.makedirs(synthesized_dir, exist_ok=True)

# Number of images to generate per class
images_per_class = 550

print(f"=== Synthesizing {images_per_class} HIGH-RESOLUTION 128x128 images per class ===")
print(f"Total images to generate: {images_per_class * class_num}")
print(f"Image resolution: {img_size}x{img_size} pixels")
print(f"Saving to: {synthesized_dir}")
print(f"Filename format: (class_label)_(image_number)_(random_5digits).png")

# Set generator to evaluation mode
generator.eval()

with torch.no_grad():
    for class_idx in range(class_num):
        print(f"\nGenerating {img_size}x{img_size} images for {class_list[class_idx]} (Class {class_idx})...")
        
        for img_num in range(images_per_class):
            # Generate random noise
            z = torch.randn(1, z_size).to(device)
            
            # Create label tensor
            label = torch.LongTensor([class_idx]).to(device)
            
            # Generate high-resolution image
            fake_image = generator(z, label)
            
            # Generate random 5-digit number
            random_digits = random.randint(10000, 99999)
            
            # Create filename: (class_label)_(image_number)_(random_5digits).png
            filename = f"{class_idx}_{img_num:03d}_{random_digits}.png"
            filepath = os.path.join(synthesized_dir, filename)
            
            # Denormalize image from [-1, 1] to [0, 1] and save
            save_image(fake_image, filepath, normalize=True, value_range=(-1, 1))
            
            # Progress indicator
            if (img_num + 1) % 50 == 0:
                print(f"  Generated {img_num + 1}/{images_per_class} high-res images for Class {class_idx}")

print(f"\n=== HIGH-RESOLUTION SYNTHESIS COMPLETE ===")
print(f"Successfully generated {images_per_class * class_num} synthetic 128x128 third molar images")
print(f"Images saved in: {synthesized_dir}")
print(f"Resolution: {img_size}x{img_size} pixels per image")

# Verify saved files
saved_files = os.listdir(synthesized_dir)
print(f"Files created: {len(saved_files)}")

# Show distribution by class
class_counts = {}
for filename in saved_files:
    if filename.endswith('.png'):
        class_label = filename.split('_')[0]
        class_counts[class_label] = class_counts.get(class_label, 0) + 1

print("\nHigh-resolution files per class:")
for class_idx in range(class_num):
    count = class_counts.get(str(class_idx), 0)
    print(f"  Class {class_idx} ({class_list[class_idx]}): {count} images (128x128)")

print(f"\nExample filenames:")
sample_files = sorted([f for f in saved_files if f.endswith('.png')])[:10]
for filename in sample_files:
    print(f"  {filename}")

print(f"\n🎯 All synthetic images are now high-resolution 128x128 pixels!")